In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import pandas as pd

file_path = "/content/drive/MyDrive/2025-2026_-_4th_Quarter_WEB.xlsx - Prov TOP30 stations.csv"

In [33]:
df = pd.read_csv(file_path, skiprows=12)

**Exploratory Data Analysis (EDA)**

Run this cell to inspect the raw structure, data types, missing values, and high-level summary statistics of the  dataset

In [34]:
#1 view the data
df.head()

,Unnamed: 0,Unnamed: 1,Prov Position,RSA Position,Station,District,Province,January 2022 to \nMarch 2022,January 2023 to \nMarch 2023,January 2024 to \nMarch 2024,January 2025 to \nMarch 2025,January 2026 to \nMarch 2026,Count Diff,(%) Change,Unnamed: 14,Unnamed: 15
0,NaN,NaN,RSA,PHO,Sta,Dis,Prov,P1,P2,P3,P4,P5,Dif,Cha,NaN,NaN
1,NaN,NaN,1,1,Cape Town Central,City of Cape Town District,Western Cape,"2,653","3,079","3,322","3,102","2,824",-278,-9.0%,NaN,NaN
2,NaN,NaN,2,2,Mitchells Plain,City of Cape Town District,Western Cape,"1,973","2,218","2,116","1,771","1,861",90,5.1%,NaN,NaN
3,NaN,NaN,3,3,Inanda,eThekwini District,KwaZulu-Natal,"1,379","1,437","1,423","1,506","1,787",281,18.7%,NaN,NaN
4,NaN,NaN,4,4,Durban Central,eThekwini District,KwaZulu-Natal,"2,037","2,386","2,207","2,044","1,784",-260,-12.7%,NaN,NaN


In [39]:
#2 info about  the dataset
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   0 non-null      float64
 1   Unnamed: 1                   0 non-null      float64
 2   Prov Position                32 non-null     object 
 3   RSA Position                 31 non-null     object 
 4   Station                      31 non-null     object 
 5   District                     31 non-null     object 
 6   Province                     31 non-null     object 
 7   January 2022 to 
March 2022  31 non-null     object 
 8   January 2023 to 
March 2023  31 non-null     object 
 9   January 2024 to 
March 2024  31 non-null     object 
 10  January 2025 to 
March 2025  31 non-null     object 
 11  January 2026 to 
March 2026  31 non-null     object 
 12  Count Diff                   31 non-null     object 
 13  (%) Change            

In [40]:
# 3. View Column Names and Data Types
print("--- COLUMN NAMES & TYPES ---")
print(df.dtypes)

--- COLUMN NAMES & TYPES ---
Unnamed: 0                      float64
Unnamed: 1                      float64
Prov Position                    object
RSA Position                     object
Station                          object
District                         object
Province                         object
January 2022 to \nMarch 2022     object
January 2023 to \nMarch 2023     object
January 2024 to \nMarch 2024     object
January 2025 to \nMarch 2025     object
January 2026 to \nMarch 2026     object
Count Diff                       object
(%) Change                       object
Unnamed: 14                     float64
Unnamed: 15                     float64
dtype: object


In [43]:
# 3. Check for Missing Values per Column, the extect number and percentage of the missing values
print("--- MISSING VALUE COUNT ---")
missing_info = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print(missing_info)

--- MISSING VALUE COUNT ---
                              Missing Values  Percentage (%)
Unnamed: 0                                33      100.000000
Unnamed: 1                                33      100.000000
Prov Position                              1        3.030303
RSA Position                               2        6.060606
Station                                    2        6.060606
District                                   2        6.060606
Province                                   2        6.060606
January 2022 to \nMarch 2022               2        6.060606
January 2023 to \nMarch 2023               2        6.060606
January 2024 to \nMarch 2024               2        6.060606
January 2025 to \nMarch 2025               2        6.060606
January 2026 to \nMarch 2026               2        6.060606
Count Diff                                 2        6.060606
(%) Change                                 2        6.060606
Unnamed: 14                               33      100.000

In [45]:
# 4. Summary Statistics for Numeric Columns
print("--- NUMERIC SUMMARY ---")
display(df.describe())

--- NUMERIC SUMMARY ---


,Unnamed: 0,Unnamed: 1,Unnamed: 14,Unnamed: 15
count,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN


**Cleaning & Handling Missing Values**

Now that we know where the missing data and messy rows live, run this cell to clean up the DataFrame:

In [46]:
# Create a copy for cleaning
df_clean = df.copy()

In [47]:
# 1. Clean Column Headers (strip spaces, lowercase, replace spaces with underscores)
df_clean.columns = df_clean.columns.astype(str).str.strip().str.lower().str.replace(' ', '_').str.replace('\n', '')

In [48]:
# 2. Drop completely empty rows or rows missing critical station/province names
# (SAPS table footer notes typically get dropped here)
station_col = [col for col in df_clean.columns if 'station' in col][0]
province_col = [col for col in df_clean.columns if 'prov' in col][0]

df_clean = df_clean.dropna(subset=[station_col, province_col])

In [49]:
# 3. Handle Duplicate Rows (if any)
duplicates_count = df_clean.duplicated().sum()
print(f"Duplicate rows found and removed: {duplicates_count}")
df_clean = df_clean.drop_duplicates()

Duplicate rows found and removed: 0


In [50]:
# 4. Fill remaining missing numeric values with 0 (counts of 0 incidents)
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)

In [51]:
# Verify clean state
print("\n--- POST-CLEANING SUMMARY ---")
print(f"Clean Rows: {df_clean.shape[0]}, Clean Columns: {df_clean.shape[1]}")
print(f"Remaining Missing Values: {df_clean.isnull().sum().sum()}")


--- POST-CLEANING SUMMARY ---
Clean Rows: 31, Clean Columns: 16
Remaining Missing Values: 0


In [53]:
df_clean.head()

,unnamed:_0,unnamed:_1,prov_position,rsa_position,station,district,province,january_2022_to_march_2022,january_2023_to_march_2023,january_2024_to_march_2024,january_2025_to_march_2025,january_2026_to_march_2026,count_diff,(%)_change,unnamed:_14,unnamed:_15
0,0.0,0.0,RSA,PHO,Sta,Dis,Prov,P1,P2,P3,P4,P5,Dif,Cha,0.0,0.0
1,0.0,0.0,1,1,Cape Town Central,City of Cape Town District,Western Cape,"2,653","3,079","3,322","3,102","2,824",-278,-9.0%,0.0,0.0
2,0.0,0.0,2,2,Mitchells Plain,City of Cape Town District,Western Cape,"1,973","2,218","2,116","1,771","1,861",90,5.1%,0.0,0.0
3,0.0,0.0,3,3,Inanda,eThekwini District,KwaZulu-Natal,"1,379","1,437","1,423","1,506","1,787",281,18.7%,0.0,0.0
4,0.0,0.0,4,4,Durban Central,eThekwini District,KwaZulu-Natal,"2,037","2,386","2,207","2,044","1,784",-260,-12.7%,0.0,0.0


In [54]:
# Drop any column that has 'unnamed' in its name because they had 100% missing values
unnamed_cols = [col for col in df_clean.columns if 'unnamed' in col.lower()]

df_clean = df_clean.drop(columns=unnamed_cols)

In [55]:
df_clean.head()

,prov_position,rsa_position,station,district,province,january_2022_to_march_2022,january_2023_to_march_2023,january_2024_to_march_2024,january_2025_to_march_2025,january_2026_to_march_2026,count_diff,(%)_change
0,RSA,PHO,Sta,Dis,Prov,P1,P2,P3,P4,P5,Dif,Cha
1,1,1,Cape Town Central,City of Cape Town District,Western Cape,"2,653","3,079","3,322","3,102","2,824",-278,-9.0%
2,2,2,Mitchells Plain,City of Cape Town District,Western Cape,"1,973","2,218","2,116","1,771","1,861",90,5.1%
3,3,3,Inanda,eThekwini District,KwaZulu-Natal,"1,379","1,437","1,423","1,506","1,787",281,18.7%
4,4,4,Durban Central,eThekwini District,KwaZulu-Natal,"2,037","2,386","2,207","2,044","1,784",-260,-12.7%


In [56]:
# Drop the 1st row (index 0) and reset the index
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# Verify the top 5 rows
df_clean.head()

,prov_position,rsa_position,station,district,province,january_2022_to_march_2022,january_2023_to_march_2023,january_2024_to_march_2024,january_2025_to_march_2025,january_2026_to_march_2026,count_diff,(%)_change
0,1,1,Cape Town Central,City of Cape Town District,Western Cape,"2,653","3,079","3,322","3,102","2,824",-278,-9.0%
1,2,2,Mitchells Plain,City of Cape Town District,Western Cape,"1,973","2,218","2,116","1,771","1,861",90,5.1%
2,3,3,Inanda,eThekwini District,KwaZulu-Natal,"1,379","1,437","1,423","1,506","1,787",281,18.7%
3,4,4,Durban Central,eThekwini District,KwaZulu-Natal,"2,037","2,386","2,207","2,044","1,784",-260,-12.7%
4,5,5,Mfuleni,City of Cape Town District,Western Cape,"1,433","1,817","1,953","1,938","1,691",-247,-12.7%


In [57]:
# List of count columns to convert to numbers
numeric_cols = [
    'january_2022_to_march_2022',
    'january_2023_to_march_2023',
    'january_2024_to_march_2024',
    'january_2025_to_march_2025',
    'january_2026_to_march_2026',
    'count_diff',
    '(%)_change'
]

# Remove commas and convert to float/int
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.replace(',', '').str.strip()
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

# Check data types to confirm they are numeric
df_clean[numeric_cols].dtypes

,0
january_2022_to_march_2022,int64
january_2023_to_march_2023,int64
january_2024_to_march_2024,int64
january_2025_to_march_2025,int64
january_2026_to_march_2026,int64
count_diff,int64
(%)_change,float64


**PREPROCESSSING**

In [58]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [59]:
# 1. Select Features for ML Models
# ---------------------------------------------------------
# Crime period features across historical quarters
feature_cols = [
    'january_2022_to_march_2022',
    'january_2023_to_march_2023',
    'january_2024_to_march_2024',
    'january_2025_to_march_2025',
    'january_2026_to_march_2026'
]

In [60]:
# Separate feature matrix X
X = df_clean[feature_cols].copy()

In [61]:
# 2. Feature Scaling (Essential for K-Means & DBSCAN)
# ---------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [62]:
# Convert back to DataFrame for easy inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=[f"{col}_scaled" for col in feature_cols])

In [63]:
# 3. Create Target Variable (Risk Level Tiers for Supervised Models)
# ---------------------------------------------------------
# Bin most recent period (2026) into 3 ordinal risk levels: Low, Medium, High Risk
df_clean['risk_level'] = pd.qcut(
    df_clean['january_2026_to_march_2026'],
    q=3,
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

In [66]:
#4 Encode 'Low Risk' -> 0, 'Medium Risk' -> 1, 'High Risk' -> 2
le = LabelEncoder()
y = le.fit_transform(df_clean['risk_level'])
df_clean['risk_level_encoded'] = y

In [67]:
# 5. Train / Test Split for Supervised Classifiers
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [68]:
print("="*50)
print("SUCCESS: Target 'y' defined and Train/Test Split complete!")
print("="*50)
print(f"X_scaled shape: {X_scaled.shape}")
print(f"Target 'y' shape: {y.shape}")
print(f"Train set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")

SUCCESS: Target 'y' defined and Train/Test Split complete!
X_scaled shape: (30, 5)
Target 'y' shape: (30,)
Train set: 24 rows | Test set: 6 rows
